In [1]:
import numpy as np
import re
import torch.nn.functional as F

def is_float(s):
    try:
        float(s)
        return True
    except Exception:
        return False

def jcamp_read(filehandle):
    """
    Robust JCAMP-DX reader for NIST and similar files.
    Handles XYDATA, compressed (X++(Y..Y)), and uncompressed data.
    """
    jcamp_dict = {}
    x = []
    y = []
    xfactor = 1.0
    yfactor = 1.0
    in_data = False
    data_format = None
    deltax = None
    last_x_value = None # New state variable for compressed data

    for line in filehandle:
        if hasattr(line, 'decode'):
            line = line.decode('utf-8', 'ignore')
        line = line.strip()
        if not line or line.startswith('$$'):
            continue

        # Header lines
        if line.startswith('##'):
            key, _, value = line[2:].partition('=')
            key = key.strip().lower()
            value = value.strip()
            jcamp_dict[key] = value
            
            if key == 'xfactor':
                try: xfactor = float(value)
                except Exception: xfactor = 1.0
            if key == 'yfactor':
                try: yfactor = float(value)
                except Exception: yfactor = 1.0
            if key == 'deltax':
                try: deltax = float(value)
                except Exception: deltax = None
            if key in ('xydata', 'peak table', 'xypoints'):
                in_data = True
                data_format = value.lower()
                continue
            if key == 'end':
                in_data = False
                continue
            continue

        # Data lines
        if in_data:
            if line.startswith('##') or line.startswith('$$'):
                continue
            
            if data_format.startswith('(x++(y..y))'):
                
                # Use simple Python split on whitespace
                tokens = [t for t in line.split() if t]
                
                if not tokens:
                    continue
                try:
                    
                    if not x: # First line of the data block
                        # Expects X0 Y1 Y2 Y3... format
                        if len(tokens) < 2: continue
                        
                        x0 = float(tokens[0]) * xfactor
                        yvals = [float(val) * yfactor for val in tokens[1:] if is_float(val)]
                        
                        # Set initial X value and point
                        x.append(x0)
                        y.append(yvals[0])
                        yvals.pop(0)
                        
                        last_x_value = x0
                        
                    else: # Subsequent lines only contain Y values
                        # Assume all tokens are Y values; X continues from last_x_value
                        yvals = [float(val) * yfactor for val in tokens if is_float(val)]
                    
                    if deltax is None:
                        # Fallback for DELTAX, should use 5.0 for IR if header is missing
                        deltax = 5.0
                    
                    # Append all points using deltax
                    for yval in yvals:
                        last_x_value += deltax
                        x.append(last_x_value)
                        y.append(yval)
                        
                except Exception as e:
                    # Catch and print the error for debugging.
                    # print(f"DEBUG: Parsing failure on line: {line}. Error: {e}") 
                    continue 
            
            # Note: Other data formats (e.g., XYPOINTS) would need their own logic here

    if len(x) == 0 or len(y) == 0:
        raise ValueError("No data found in JCAMP file or unable to parse x/y arrays.")

    jcamp_dict['x'] = np.array(x)
    jcamp_dict['y'] = np.array(y)
    return jcamp_dict

def jcamp_readfile(filename):
    with open(filename, 'rb') as filehandle:
        return jcamp_read(filehandle)

In [2]:
def convert_x(x_in, unit_from, unit_to):
    """Convert between micrometer and wavenumber."""
    if unit_to == 'micrometers' and unit_from == 'MICROMETERS':
        x_out = x_in
        return x_out
    elif unit_to == 'cm-1' and unit_from in ['1/CM', 'cm-1', '1/cm', 'Wavenumbers (cm-1)']:
        x_out = x_in
        return x_out
    elif unit_to == 'micrometers' and unit_from in ['1/CM', 'cm-1', '1/cm', 'Wavenumbers (cm-1)']:
        x_out = np.array([10 ** 4 / i for i in x_in])
        return x_out
    elif unit_to == 'cm-1' and unit_from == 'MICROMETERS':
        x_out = np.array([10 ** 4 / i for i in x_in])
        return x_out

def convert_y(y_in, unit_from, unit_to):
    """Convert between absorbance and trasmittance."""
    if unit_to == 'transmittance' and unit_from in ['% Transmission', 'TRANSMITTANCE', 'Transmittance']:
        y_out = y_in
        return y_out
    elif unit_to == 'absorbance' and unit_from == 'ABSORBANCE':
        y_out = y_in
        return y_out
    elif unit_to == 'transmittance' and unit_from == 'ABSORBANCE':
        y_out = np.array([1 / 10 ** j for j in y_in])
        return y_out
    elif unit_to == 'absorbance' and unit_from in ['% Transmission', 'TRANSMITTANCE', 'Transmittance']:
        y_out = np.array([np.log10(1 / j) for j in y_in])
        return y_out

def get_unique(x_in, y_in):
    """Removes duplicates in x and takes smallest y value for each x value."""
    x_out = sorted(list(set(x_in)), reverse=True)
    y_out = []
    for i in x_out:
        y_temp = []
        for ii, j in zip(x_in, y_in):
            if i == ii:
                y_temp.append(j)
        y_out.append(min(y_temp))
    return x_out, y_out

import numpy as np

def convert_transmittance_to_absorbance(T_data: np.ndarray, base=10) -> np.ndarray:
    """
    Converts Transmittance (T) values to Absorbance (A) values.
    
    A = -log10(T)
    
    Args:
        T_data: NumPy array of Transmittance values (T, between 0 and 1).
        base: Logarithm base (10 for IR spectroscopy).
        
    Returns:
        NumPy array of Absorbance values (A).
    """
    # Protect against T=0, which causes log10(0) = -inf
    # Use a small epsilon (1e-6) if T is close to zero.
    epsilon = 1e-6
    T_clamped = np.clip(T_data, epsilon, 1.0)
    
    # Calculate Absorbance (A = -log10(T))
    A_data = -np.log10(T_clamped)
    
    return A_data

In [3]:
from os import listdir
from scipy import interpolate
from rdkit import Chem
from smarts import fg_list_original, fg_list_extended
from PIL import Image
import os
import numpy as np
import sys
import csv
import cv2
import re
import pandas as pd

jdx_path = r"D:\IITM Intern\irchracterizationcnn--main (3) 2\irchracterizationcnn--main (3)\irchracterizationcnn--main\scripts\Codebase for Github\Scraping Codes\nist_data\jdx"
out_csv_path = r"D:\IITM Intern\irchracterizationcnn--main (3) 2\irchracterizationcnn--main (3)\irchracterizationcnn--main\scripts\Codebase for Github\Scraping Codes\nist_data"
os.makedirs(out_csv_path, exist_ok=True)

combined_csv_path = os.path.join(out_csv_path, "nist_dataset.csv")

skipped = []
processed = 0
combined_rows = []

# fixed output grid used for all molecules
x_grid = np.linspace(4000, 400, 1648)

for file in listdir(jdx_path):
    file_path = os.path.join(jdx_path, file)
    molecule_id = os.path.splitext(file)[0]

    try:
        spectrum = jcamp_readfile(file_path)
    except Exception as e:
        skipped.append((file, f"read_error: {e}"))
        continue

    try:
        jcamp_dict = spectrum
        x = jcamp_dict['x']
        y = jcamp_dict['y']

        xunit = jcamp_dict.get('xunits', jcamp_dict.get('xlabel', '1/CM'))
        yunit = jcamp_dict.get('yunits', jcamp_dict.get('ylabel', 'TRANSMITTANCE'))

        x = convert_x(x, xunit, 'cm-1')
        y = convert_y(y, yunit, 'transmittance')

        y_min = min(y)
        y_max = max(y)
        x_min = min(x)
        x_max = max(x)

        if y_max > 1:
            y = [1 if j > 1 else j for j in y]
        if y_min < 0:
            y = [0 if j < 0 else j for j in y]

        if x[0] > x[1]:
            x = x[::-1]
            y = y[::-1]

        if x_max > 4000:
            idx = next(i for i, xx in enumerate(x) if xx >= 4000)
            x = x[:idx + 1]
            y = y[:idx + 1]
        if x_min < 400:
            idx = next(i for i, xx in enumerate(x) if xx >= 400)
            x = x[idx - 1:]
            y = y[idx - 1:]

        if x_max < 4000:
            x = np.append(x, [x[-1] + 1, 4000])
            y = np.append(y, [y[-1], y[-1]])
        if x_min > 400:
            x = np.insert(x, 0, x[0] - 1)
            x = np.insert(x, 0, 400)
            y = np.insert(y, 0, y[0])
            y = np.insert(y, 0, y[0])

        spectrum = get_unique(x, y)
        f = interpolate.interp1d(spectrum[0], spectrum[1], kind='cubic')
        y_interp = f(x_grid)

        y_abs = convert_transmittance_to_absorbance(y_interp)

        # one row per molecule: [molecule_id, 1648 absorbance values]
        combined_rows.append([molecule_id] + list(y_abs))
        processed += 1

    except Exception as e:
        skipped.append((file, f"process_error: {e}"))
        continue

# Save one combined CSV (each row = one molecule)
feature_cols = [f"{i}" for i in range(len(x_grid))]
combined_df = pd.DataFrame(combined_rows, columns=["molecule_id"] + feature_cols)
combined_df.to_csv(combined_csv_path, index=False)

print(f"Processed: {processed}")
print(f"Skipped: {len(skipped)}")
print(f"Combined CSV saved: {combined_csv_path}")
for name, reason in skipped[:20]:
    print(f" - {name}: {reason}")


Processed: 47
Skipped: 3
Combined CSV saved: D:\IITM Intern\irchracterizationcnn--main (3) 2\irchracterizationcnn--main (3)\irchracterizationcnn--main\scripts\Codebase for Github\Scraping Codes\nist_data\nist_dataset.csv
 - B6000063_1.jdx: read_error: No data found in JCAMP file or unable to parse x/y arrays.
 - B6000094_1.jdx: read_error: No data found in JCAMP file or unable to parse x/y arrays.
 - B6000241_0.jdx: read_error: No data found in JCAMP file or unable to parse x/y arrays.
